# Exploratory Data Analysis — MIMIC-III ICU Data

**Smart ICU Assistant** | Demographics, Distributions, Labels, Correlations & Temporal Analysis

This notebook generates **13 publication-ready figures** from MIMIC-III for the project report.

| # | Figure | Section |
|---|--------|----------|
| 1 | Dataset overview table | Demographics |
| 2 | Age distribution (survivors vs non-survivors) | Demographics |
| 3 | Gender distribution with mortality overlay | Demographics |
| 4 | ICU Length of Stay distribution | Demographics |
| 5 | Care unit distribution | Demographics |
| 6 | Vital signs box plots | Distributions |
| 7 | Lab values box plots | Distributions |
| 8 | Label prevalence / class imbalance | Labels |
| 9 | Feature correlation heatmap | Correlations |
| 10 | Missing data analysis | Data Quality |
| 11 | Mortality by age group | Clinical |
| 12 | Temporal vital trends | Temporal |
| 13 | Admission type distribution | Clinical |


## Setup & Configuration

In [1]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch
from tqdm import tqdm
import logging

from data_loader import MIMICDataLoader

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# ── Style ──
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.family': 'sans-serif',
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

PALETTE = sns.color_palette('deep')
COLOR_SURV = '#2196F3'
COLOR_DEAD = '#F44336'
COLOR_MALE = '#42A5F5'
COLOR_FEMALE = '#EF5350'
OUTPUT_DIR = os.path.join('output', 'eda')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLE_N = 5000        # vitals/labs sampling
SAMPLE_N_TEMPORAL = 3000  # temporal analysis sampling


## Load MIMIC-III Data

In [2]:
print("Loading MIMIC-III data...")
loader = MIMICDataLoader("data", "config.yaml")
merged = loader.merge_data()
print(f"Loaded {len(merged):,} ICU stays")


Loading MIMIC-III data...


  Data loading:   0%|          | 0/17 [00:00<?, ?step/s]INFO: ============================================================
INFO: DATA LOADING PIPELINE
INFO: ============================================================
INFO: PATIENTS: 46,520 rows
  DICTIONARIES ✓                     :  18%|█▊        | 3/17 [00:01<00:06,  2.09step/s]INFO: CHARTEVENTS is 32.9 GB — starting chunked load (filtering to vital/vent itemids only)
INFO:   Filtering to 32 relevant itemids
INFO: CHARTEVENTS: kept 42,342,726 rows (from ~330,712,483 total)
  CHARTEVENTS ✓                      :  29%|██▉       | 5/17 [11:36<39:23, 196.99s/step]INFO: Assigning icustay_id to lab events via time overlap...
INFO: LABEVENTS: 13,211,132 rows (in-stay)
  SERVICES ✓                         :  94%|█████████▍| 16/17 [13:29<00:06,  6.58s/step]INFO: Merging tables...
INFO:   Clamped 1 ages >89 → 91.4 (MIMIC-III de-id)
  MERGE ✓                            : 100%|██████████| 17/17 [13:30<00:00, 47.69s/step]
INFO: Merged dataset: 6

Loaded 61,532 ICU stays


---
## Part 1 — Demographics & Distributions

### Figure 1 — Dataset Overview Table

In [3]:
def fig_dataset_overview(merged: pd.DataFrame, loader: MIMICDataLoader):
    """Styled table summarizing dataset dimensions."""
    logger.info("Figure 1: Dataset overview table")

    n_patients = merged['subject_id'].nunique()
    n_admissions = merged['hadm_id'].nunique()
    n_stays = len(merged)
    pct_male = (merged['gender'] == 'M').mean() * 100
    mortality = merged['expire_flag'].mean() * 100
    median_age = merged['age'].median()
    mean_age = merged['age'].mean()
    median_los = merged['los'].median() if 'los' in merged.columns else np.nan

    # Compute median LOS from intime/outtime if 'los' column missing
    if pd.isna(median_los):
        los_hours = (merged['outtime'] - merged['intime']).dt.total_seconds() / 3600
        median_los_h = los_hours.median()
        los_str = f"{median_los_h:.1f} hours"
    else:
        los_str = f"{median_los:.1f} days"

    chart_rows = len(loader.chartevents) if loader.chartevents is not None else 0
    lab_rows = len(loader.labevents) if loader.labevents is not None else 0

    rows = [
        ['Total Dataset Size', '43.6 GB (uncompressed)'],
        ['Unique Patients', f'{n_patients:,}'],
        ['Hospital Admissions', f'{n_admissions:,}'],
        ['ICU Stays', f'{n_stays:,}'],
        ['Median Age', f'{median_age:.1f} years'],
        ['Mean Age', f'{mean_age:.1f} years'],
        ['Male Patients', f'{pct_male:.1f}%'],
        ['In-Hospital Mortality', f'{mortality:.1f}%'],
        ['Median ICU LOS', los_str],
        ['CHARTEVENTS Rows (filtered)', f'{chart_rows:,}'],
        ['LABEVENTS Rows (in-stay)', f'{lab_rows:,}'],
        ['Engineered Features / Timestep', '81'],
        ['Prediction Labels', '19'],
    ]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axis('off')
    table = ax.table(
        cellText=rows,
        colLabels=['Statistic', 'Value'],
        loc='center',
        cellLoc='left',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.6)

    # Style header
    for j in range(2):
        cell = table[0, j]
        cell.set_facecolor('#1565C0')
        cell.set_text_props(color='white', weight='bold')

    # Alternate row colors
    for i in range(1, len(rows) + 1):
        for j in range(2):
            cell = table[i, j]
            cell.set_facecolor('#E3F2FD' if i % 2 == 0 else 'white')

    ax.set_title('MIMIC-III Dataset Overview', fontsize=16, fontweight='bold', pad=20)
    path = os.path.join(OUTPUT_DIR, 'eda_dataset_overview.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 2 — Age Distribution
# ═══════════════════════════════════════════════════════════════════════════════


fig_dataset_overview(merged, loader)


INFO: Figure 1: Dataset overview table
INFO:   → output\eda\eda_dataset_overview.png


### Figure 2 — Age Distribution

In [4]:
def fig_age_distribution(merged: pd.DataFrame):
    """Histogram + KDE of age, stratified by survival."""
    logger.info("Figure 2: Age distribution")

    ages = merged[['age', 'expire_flag']].dropna(subset=['age']).copy()
    ages['Outcome'] = ages['expire_flag'].map({0: 'Survived', 1: 'Deceased'})

    fig, ax = plt.subplots(figsize=(10, 6))

    # Histogram for each group
    for label, color in [('Survived', COLOR_SURV), ('Deceased', COLOR_DEAD)]:
        subset = ages[ages['Outcome'] == label]['age']
        ax.hist(subset, bins=40, alpha=0.55, color=color, label=label, edgecolor='white')

    # KDE overlay
    for label, color in [('Survived', COLOR_SURV), ('Deceased', COLOR_DEAD)]:
        subset = ages[ages['Outcome'] == label]['age']
        if len(subset) > 10:
            subset.plot.kde(ax=ax, color=color, linewidth=2, label=f'{label} (KDE)')

    # Reference lines
    median_age = ages['age'].median()
    mean_age = ages['age'].mean()
    ax.axvline(median_age, color='#FF9800', linestyle='--', linewidth=2,
               label=f'Median = {median_age:.1f}')
    ax.axvline(mean_age, color='#9C27B0', linestyle=':', linewidth=2,
               label=f'Mean = {mean_age:.1f}')

    ax.set_xlabel('Age (years)')
    ax.set_ylabel('Count')
    ax.set_title('Age Distribution: Survivors vs Deceased')
    ax.legend(framealpha=0.9)
    ax.set_xlim(15, 95)

    path = os.path.join(OUTPUT_DIR, 'eda_age_distribution.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 3 — Gender Distribution
# ═══════════════════════════════════════════════════════════════════════════════


fig_age_distribution(merged)


INFO: Figure 2: Age distribution
INFO:   → output\eda\eda_age_distribution.png


### Figure 3 — Gender Distribution

In [5]:
def fig_gender_distribution(merged: pd.DataFrame):
    """Bar chart of gender counts with mortality rate overlay."""
    logger.info("Figure 3: Gender distribution")

    gender_stats = merged.groupby('gender').agg(
        count=('subject_id', 'count'),
        mortality=('expire_flag', 'mean')
    ).reset_index()
    gender_stats['mortality_pct'] = gender_stats['mortality'] * 100
    gender_stats['gender_label'] = gender_stats['gender'].map({'M': 'Male', 'F': 'Female'})

    fig, ax1 = plt.subplots(figsize=(8, 6))

    bars = ax1.bar(
        gender_stats['gender_label'],
        gender_stats['count'],
        color=[COLOR_MALE, COLOR_FEMALE],
        edgecolor='white',
        width=0.5,
        zorder=3,
    )

    # Add count labels on bars
    for bar, count in zip(bars, gender_stats['count']):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold', fontsize=13)

    ax1.set_ylabel('Number of ICU Stays', fontsize=12)
    ax1.set_title('Gender Distribution with Mortality Rate', fontsize=14, fontweight='bold')

    # Mortality overlay on secondary axis
    ax2 = ax1.twinx()
    ax2.plot(gender_stats['gender_label'], gender_stats['mortality_pct'],
             'D-', color='#F44336', markersize=10, linewidth=2, label='Mortality Rate')
    ax2.set_ylabel('Mortality Rate (%)', color='#F44336', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='#F44336')
    ax2.set_ylim(0, max(gender_stats['mortality_pct']) * 1.5)
    ax2.legend(loc='upper right')

    path = os.path.join(OUTPUT_DIR, 'eda_gender_distribution.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 4 — ICU Length of Stay
# ═══════════════════════════════════════════════════════════════════════════════


fig_gender_distribution(merged)


INFO: Figure 3: Gender distribution
INFO:   → output\eda\eda_gender_distribution.png


### Figure 4 — ICU Length of Stay

In [6]:
def fig_los_distribution(merged: pd.DataFrame):
    """Histogram of ICU LOS with clinical thresholds marked."""
    logger.info("Figure 4: Length of stay distribution")

    los_hours = (merged['outtime'] - merged['intime']).dt.total_seconds() / 3600
    los_hours = los_hours[los_hours > 0]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Left: linear scale (capped at 500h for readability)
    los_capped = los_hours.clip(upper=500)
    ax1.hist(los_capped, bins=60, color='#26A69A', edgecolor='white', alpha=0.8)
    ax1.axvline(24, color='#FF9800', linestyle='--', linewidth=2, label='24h (Short Stay)')
    ax1.axvline(72, color='#F44336', linestyle='--', linewidth=2, label='72h (Long Stay)')
    ax1.axvline(los_hours.median(), color='#9C27B0', linestyle=':', linewidth=2,
                label=f'Median = {los_hours.median():.1f}h')
    ax1.set_xlabel('ICU Length of Stay (hours)')
    ax1.set_ylabel('Count')
    ax1.set_title('ICU LOS Distribution (Linear Scale)')
    ax1.legend(fontsize=9)

    # Right: log scale (full range)
    ax2.hist(los_hours, bins=80, color='#26A69A', edgecolor='white', alpha=0.8)
    ax2.set_yscale('log')
    ax2.axvline(24, color='#FF9800', linestyle='--', linewidth=2, label='24h')
    ax2.axvline(72, color='#F44336', linestyle='--', linewidth=2, label='72h')
    ax2.set_xlabel('ICU Length of Stay (hours)')
    ax2.set_ylabel('Count (log scale)')
    ax2.set_title('ICU LOS Distribution (Log Scale)')
    ax2.legend(fontsize=9)

    # Add stats text box
    stats_text = (
        f"N = {len(los_hours):,}\n"
        f"Mean = {los_hours.mean():.1f}h\n"
        f"Median = {los_hours.median():.1f}h\n"
        f"<24h: {(los_hours < 24).mean()*100:.1f}%\n"
        f">72h: {(los_hours > 72).mean()*100:.1f}%"
    )
    ax2.text(0.97, 0.95, stats_text, transform=ax2.transAxes, fontsize=9,
             va='top', ha='right', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.suptitle('ICU Length of Stay Analysis', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'eda_los_distribution.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 5 — Care Unit Distribution
# ═══════════════════════════════════════════════════════════════════════════════


fig_los_distribution(merged)


INFO: Figure 4: Length of stay distribution
INFO:   → output\eda\eda_los_distribution.png


### Figure 5 — Care Unit Distribution

In [7]:
def fig_care_units(merged: pd.DataFrame):
    """Horizontal bar chart of ICU care units."""
    logger.info("Figure 5: Care unit distribution")

    col = None
    for c in ['first_careunit', 'curr_careunit', 'last_careunit']:
        if c in merged.columns:
            col = c
            break
    if col is None:
        logger.warning("  No care unit column found, skipping")
        return

    unit_counts = merged[col].value_counts().sort_values(ascending=True)

    # Compute mortality per unit
    unit_mort = merged.groupby(col)['expire_flag'].mean() * 100

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = sns.color_palette('Blues_d', len(unit_counts))
    bars = ax.barh(unit_counts.index, unit_counts.values, color=colors, edgecolor='white')

    # Add count + mortality labels
    for i, (unit, count) in enumerate(unit_counts.items()):
        mort = unit_mort.get(unit, 0)
        ax.text(count + max(unit_counts) * 0.01, i,
                f' {count:,}  (mort: {mort:.1f}%)',
                va='center', fontsize=10)

    ax.set_xlabel('Number of ICU Stays')
    ax.set_title('ICU Care Unit Distribution', fontsize=14, fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{int(x):,}'))

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'eda_care_units.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 6 — Vital Signs Box Plots
# ═══════════════════════════════════════════════════════════════════════════════


fig_care_units(merged)


INFO: Figure 5: Care unit distribution
INFO:   → output\eda\eda_care_units.png


### Figure 6 — Vital Signs Box Plots

In [8]:
def fig_vitals_boxplots(loader: MIMICDataLoader, sample_n=5000):
    """Box plots of vital signs with clinical reference ranges."""
    logger.info(f"Figure 6: Vital signs distributions (sampling {sample_n} stays)")

    if loader.chartevents is None or len(loader.chartevents) == 0:
        logger.warning("  No chartevents loaded, skipping")
        return

    vital_itemids = {
        'Heart Rate': [220045, 211],
        'Systolic BP': [220050, 220179, 51, 455],
        'Diastolic BP': [220051, 220180, 8368, 8441],
        'Resp Rate': [220210, 224690, 618, 615],
        'SpO₂ (%)': [220277, 646],
        'Temp (°C)': [223761, 223762, 676, 678],
        'Glucose': [220621, 226537, 807, 811, 1529],
    }

    # Sample stays
    all_stays = loader.chartevents['icustay_id'].dropna().unique()
    if len(all_stays) > sample_n:
        sampled = np.random.choice(all_stays, sample_n, replace=False)
        charts = loader.chartevents[loader.chartevents['icustay_id'].isin(sampled)]
    else:
        charts = loader.chartevents

    # Collect distributions
    data_frames = []
    for name, ids in vital_itemids.items():
        vals = charts[charts['itemid'].isin(ids)]['valuenum'].dropna()
        if len(vals) == 0:
            continue
        # Remove extreme outliers (0.5th and 99.5th percentile)
        lo, hi = vals.quantile(0.005), vals.quantile(0.995)
        vals = vals[(vals >= lo) & (vals <= hi)]
        df = pd.DataFrame({'Vital Sign': name, 'Value': vals.values})
        data_frames.append(df)

    if not data_frames:
        logger.warning("  No vital data found")
        return

    all_vitals = pd.concat(data_frames, ignore_index=True)

    # Separate plots for different scales
    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    axes = axes.flatten()

    # Clinical reference ranges (normal)
    ref_ranges = {
        'Heart Rate': (60, 100),
        'Systolic BP': (90, 140),
        'Diastolic BP': (60, 90),
        'Resp Rate': (12, 20),
        'SpO₂ (%)': (94, 100),
        'Temp (°C)': (36.0, 38.3),
        'Glucose': (70, 180),
    }

    for i, name in enumerate(vital_itemids.keys()):
        ax = axes[i]
        subset = all_vitals[all_vitals['Vital Sign'] == name]['Value']
        if len(subset) == 0:
            ax.set_visible(False)
            continue

        bp = ax.boxplot(subset.values, vert=True, patch_artist=True,
                        boxprops=dict(facecolor='#42A5F5', alpha=0.7),
                        medianprops=dict(color='#F44336', linewidth=2),
                        whiskerprops=dict(color='#666'),
                        flierprops=dict(marker='.', markersize=2, alpha=0.3))

        # Add reference range band
        if name in ref_ranges:
            lo, hi = ref_ranges[name]
            ax.axhspan(lo, hi, alpha=0.15, color='#4CAF50', label='Normal')

        ax.set_title(name, fontweight='bold')
        ax.set_xticklabels([f'n={len(subset):,}'], fontsize=8)
        ax.grid(axis='y', alpha=0.3)

        # Stats annotation
        ax.text(0.02, 0.98,
                f'μ={subset.mean():.1f}\nσ={subset.std():.1f}\nmed={subset.median():.1f}',
                transform=ax.transAxes, fontsize=7, va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # Hide last subplot if unused
    if len(vital_itemids) < len(axes):
        for j in range(len(vital_itemids), len(axes)):
            axes[j].set_visible(False)

    plt.suptitle('Vital Signs Distributions (with Normal Ranges)',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'eda_vitals_boxplots.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 7 — Lab Values Box Plots
# ═══════════════════════════════════════════════════════════════════════════════


fig_vitals_boxplots(loader, sample_n=SAMPLE_N)


INFO: Figure 6: Vital signs distributions (sampling 5000 stays)
C:\Users\ncfad\AppData\Local\Temp\ipykernel_26668\736623150.py:95: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\ncfad\AppData\Local\Temp\ipykernel_26668\736623150.py:97: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig(path)
INFO:   → output\eda\eda_vitals_boxplots.png


### Figure 7 — Lab Values Box Plots

In [9]:
def fig_labs_boxplots(loader: MIMICDataLoader, sample_n=5000):
    """Box plots of laboratory values with clinical reference ranges."""
    logger.info(f"Figure 7: Lab values distributions (sampling {sample_n} stays)")

    if loader.labevents is None or len(loader.labevents) == 0:
        logger.warning("  No labevents loaded, skipping")
        return

    # Build lab itemid mapping
    lab_keywords = {
        'Creatinine': 'creatinine',
        'Lactate': 'lactate',
        'WBC': 'wbc',
        'Hemoglobin': 'hemoglobin',
        'Platelets': 'platelet',
        'Bicarbonate': 'bicarbonate',
        'Chloride': 'chloride',
    }

    lab_itemids = {}
    if loader.d_labitems is not None:
        for display_name, keyword in lab_keywords.items():
            mask = loader.d_labitems['label'].str.lower().str.contains(keyword, na=False)
            ids = loader.d_labitems[mask]['itemid'].unique().tolist()
            if ids:
                lab_itemids[display_name] = ids

    if not lab_itemids:
        logger.warning("  No lab mappings found")
        return

    # Sample stays
    all_stays = loader.labevents['icustay_id'].dropna().unique()
    if len(all_stays) > sample_n:
        sampled = np.random.choice(all_stays, sample_n, replace=False)
        labs = loader.labevents[loader.labevents['icustay_id'].isin(sampled)]
    else:
        labs = loader.labevents

    # Clinical reference ranges
    ref_ranges = {
        'Creatinine': (0.6, 1.2),
        'Lactate': (0.5, 2.0),
        'WBC': (4.0, 12.0),
        'Hemoglobin': (12.0, 17.5),
        'Platelets': (150, 400),
        'Bicarbonate': (22, 28),
        'Chloride': (96, 106),
    }

    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    axes = axes.flatten()

    for i, (name, ids) in enumerate(lab_itemids.items()):
        ax = axes[i]
        vals = labs[labs['itemid'].isin(ids)]['valuenum'].dropna()
        if len(vals) == 0:
            ax.set_visible(False)
            continue

        # Remove extreme outliers
        lo, hi = vals.quantile(0.005), vals.quantile(0.995)
        vals = vals[(vals >= lo) & (vals <= hi)]

        bp = ax.boxplot(vals.values, vert=True, patch_artist=True,
                        boxprops=dict(facecolor='#66BB6A', alpha=0.7),
                        medianprops=dict(color='#F44336', linewidth=2),
                        whiskerprops=dict(color='#666'),
                        flierprops=dict(marker='.', markersize=2, alpha=0.3))

        if name in ref_ranges:
            lo_r, hi_r = ref_ranges[name]
            ax.axhspan(lo_r, hi_r, alpha=0.15, color='#4CAF50', label='Normal')

        ax.set_title(name, fontweight='bold')
        ax.set_xticklabels([f'n={len(vals):,}'], fontsize=8)
        ax.grid(axis='y', alpha=0.3)

        ax.text(0.02, 0.98,
                f'μ={vals.mean():.2f}\nσ={vals.std():.2f}\nmed={vals.median():.2f}',
                transform=ax.transAxes, fontsize=7, va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    for j in range(len(lab_itemids), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Laboratory Values Distributions (with Normal Ranges)',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'eda_labs_boxplots.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════════════════════


fig_labs_boxplots(loader, sample_n=SAMPLE_N)


INFO: Figure 7: Lab values distributions (sampling 5000 stays)
INFO:   → output\eda\eda_labs_boxplots.png


---
## Part 2 — Labels, Correlations & Temporal Analysis

### Figure 8 — Label Prevalence

In [10]:
def fig_label_prevalence():
    """Bar chart showing positive rate for all 19 prediction labels."""
    logger.info("Figure 8: Label prevalence / class imbalance")

    y_path = os.path.join('output', 'feature_cache_y.npy')
    meta_path = os.path.join('output', 'feature_cache_meta.pkl')

    if not os.path.exists(y_path) or not os.path.exists(meta_path):
        logger.warning("  Feature cache not found — skipping label prevalence")
        return

    y = np.load(y_path)
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    label_names = meta['label_names']

    pos_rates = (y > 0.5).mean(axis=0) * 100

    # Sort by positive rate
    order = np.argsort(pos_rates)[::-1]

    fig, ax = plt.subplots(figsize=(14, 7))

    # Color by severity: red if <5%, orange if <15%, green otherwise
    colors = []
    for rate in pos_rates[order]:
        if rate < 5:
            colors.append('#F44336')
        elif rate < 15:
            colors.append('#FF9800')
        else:
            colors.append('#4CAF50')

    names = [label_names[i] for i in order]
    bars = ax.barh(range(len(names)), pos_rates[order], color=colors,
                   edgecolor='white', height=0.7)

    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=10)
    ax.set_xlabel('Positive Class Rate (%)')
    ax.set_title('Prediction Label Prevalence — Class Imbalance Analysis',
                 fontsize=14, fontweight='bold')
    ax.invert_yaxis()

    # Add percentage labels
    for i, (bar, rate) in enumerate(zip(bars, pos_rates[order])):
        ratio = 100 / rate - 1 if rate > 0 else float('inf')
        ratio_str = f'1:{ratio:.0f}' if ratio < 1000 else 'extreme'
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f' {rate:.1f}%  ({ratio_str})',
                va='center', fontsize=9)

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#F44336', label='Severe imbalance (<5%)'),
        Patch(facecolor='#FF9800', label='Moderate imbalance (5-15%)'),
        Patch(facecolor='#4CAF50', label='Manageable (>15%)'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'eda_label_prevalence.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 9 — Correlation Heatmap
# ═══════════════════════════════════════════════════════════════════════════════


fig_label_prevalence()


INFO: Figure 8: Label prevalence / class imbalance
INFO:   → output\eda\eda_label_prevalence.png


### Figure 9 — Correlation Heatmap

In [11]:
def fig_correlation_heatmap():
    """Feature-feature correlation from cached features."""
    logger.info("Figure 9: Correlation heatmap")

    x_path = os.path.join('output', 'feature_cache_X.npy')
    if not os.path.exists(x_path):
        logger.warning("  Feature cache X not found — skipping correlation")
        return

    # Load with mmap and sample for speed
    X = np.load(x_path, mmap_mode='r')
    n_samples = min(10000, X.shape[0])
    idx = np.random.choice(X.shape[0], n_samples, replace=False)

    # Average across time steps to get [samples × features]
    X_avg = np.nanmean(X[idx], axis=1)
    n_feat = X_avg.shape[1]

    # Only show first 20 features for readability
    n_show = min(20, n_feat)
    X_sub = X_avg[:, :n_show]

    # Feature names (approximate — from config order)
    feat_names = [
        'HR_mean', 'HR_std', 'HR_min', 'HR_max', 'HR_trend',
        'SBP_mean', 'SBP_std', 'SBP_min', 'SBP_max', 'SBP_trend',
        'DBP_mean', 'DBP_std', 'DBP_min', 'DBP_max', 'DBP_trend',
        'MAP_mean', 'MAP_std', 'MAP_min', 'MAP_max', 'MAP_trend',
    ][:n_show]

    corr = np.corrcoef(X_sub.T)
    np.fill_diagonal(corr, 1.0)  # ensure diagonal is exactly 1

    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1,
                xticklabels=feat_names, yticklabels=feat_names,
                square=True, linewidths=0.5, ax=ax,
                annot_kws={'size': 7},
                cbar_kws={'label': 'Pearson Correlation', 'shrink': 0.8})

    ax.set_title('Feature Correlation Heatmap (First 20 Features)',
                 fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(fontsize=9)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'eda_correlation_heatmap.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 10 — Missing Data Analysis
# ═══════════════════════════════════════════════════════════════════════════════


fig_correlation_heatmap()


INFO: Figure 9: Correlation heatmap
INFO:   → output\eda\eda_correlation_heatmap.png


### Figure 10 — Missing Data Analysis

In [12]:
def fig_missingness(loader: MIMICDataLoader, sample_n=3000):
    """Bar chart showing % missing per vital/lab feature."""
    logger.info(f"Figure 10: Missing data analysis (sampling {sample_n} stays)")

    if loader.icu_stays is None or len(loader.icu_stays) == 0:
        logger.warning("  No ICU stays, skipping")
        return

    vital_itemids = {
        'Heart Rate': [220045, 211],
        'Systolic BP': [220050, 220179, 51, 455],
        'Diastolic BP': [220051, 220180, 8368, 8441],
        'Mean ABP': [220052, 220181, 52, 456],
        'Resp Rate': [220210, 224690, 618, 615],
        'SpO₂': [220277, 646],
        'Temperature': [223761, 223762, 676, 678],
        'Glucose (chart)': [220621, 226537, 807, 811, 1529],
    }

    lab_keywords = {
        'Creatinine': 'creatinine',
        'Lactate': 'lactate',
        'WBC': 'wbc',
        'Hemoglobin': 'hemoglobin',
        'Platelets': 'platelet',
        'Bicarbonate': 'bicarbonate',
        'Chloride': 'chloride',
    }

    lab_itemids = {}
    if loader.d_labitems is not None:
        for name, kw in lab_keywords.items():
            mask = loader.d_labitems['label'].str.lower().str.contains(kw, na=False)
            ids = loader.d_labitems[mask]['itemid'].unique().tolist()
            if ids:
                lab_itemids[name] = ids

    # Sample ICU stays
    all_stays = loader.icu_stays['icustay_id'].unique()
    if len(all_stays) > sample_n:
        sampled_stays = np.random.choice(all_stays, sample_n, replace=False)
    else:
        sampled_stays = all_stays

    # Count stays with at least one reading per feature
    missing_pct = {}

    if loader.chartevents is not None and len(loader.chartevents) > 0:
        charts_sampled = loader.chartevents[
            loader.chartevents['icustay_id'].isin(sampled_stays)
        ]
        for name, ids in vital_itemids.items():
            stays_with = charts_sampled[charts_sampled['itemid'].isin(ids)]['icustay_id'].nunique()
            missing_pct[name] = (1 - stays_with / len(sampled_stays)) * 100

    if loader.labevents is not None and len(loader.labevents) > 0:
        labs_sampled = loader.labevents[
            loader.labevents['icustay_id'].isin(sampled_stays)
        ]
        for name, ids in lab_itemids.items():
            stays_with = labs_sampled[labs_sampled['itemid'].isin(ids)]['icustay_id'].nunique()
            missing_pct[name] = (1 - stays_with / len(sampled_stays)) * 100

    if not missing_pct:
        logger.warning("  No missingness data")
        return

    # Sort by missingness
    sorted_items = sorted(missing_pct.items(), key=lambda x: x[1], reverse=True)
    names = [x[0] for x in sorted_items]
    values = [x[1] for x in sorted_items]

    fig, ax = plt.subplots(figsize=(12, 7))

    colors = ['#F44336' if v > 50 else '#FF9800' if v > 20 else '#4CAF50' for v in values]
    bars = ax.barh(range(len(names)), values, color=colors, edgecolor='white', height=0.6)

    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=11)
    ax.set_xlabel('% of ICU Stays Missing This Feature')
    ax.set_title(f'Feature Missingness Analysis (n={len(sampled_stays):,} stays)',
                 fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.set_xlim(0, 105)

    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=10)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#F44336', label='High missingness (>50%)'),
        Patch(facecolor='#FF9800', label='Moderate (20-50%)'),
        Patch(facecolor='#4CAF50', label='Low (<20%)'),
    ]
    ax.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'eda_missingness.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 11 — Mortality by Age Group
# ═══════════════════════════════════════════════════════════════════════════════


fig_missingness(loader, sample_n=SAMPLE_N)


INFO: Figure 10: Missing data analysis (sampling 5000 stays)
C:\Users\ncfad\AppData\Local\Temp\ipykernel_26668\922307955.py:98: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\ncfad\AppData\Local\Temp\ipykernel_26668\922307955.py:100: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig(path)
INFO:   → output\eda\eda_missingness.png


### Figure 11 — Mortality by Age Group

In [13]:
def fig_mortality_by_age(merged: pd.DataFrame):
    """Grouped bar: mortality rate by age decile."""
    logger.info("Figure 11: Mortality by age group")

    df = merged[['age', 'expire_flag']].dropna().copy()
    bins = [0, 30, 40, 50, 60, 70, 80, 100]
    labels = ['<30', '30-39', '40-49', '50-59', '60-69', '70-79', '80+']
    df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

    stats = df.groupby('age_group', observed=True).agg(
        count=('expire_flag', 'count'),
        deaths=('expire_flag', 'sum'),
        mortality=('expire_flag', 'mean'),
    ).reset_index()
    stats['mortality_pct'] = stats['mortality'] * 100

    fig, ax1 = plt.subplots(figsize=(10, 6))

    x = range(len(stats))
    bars = ax1.bar(x, stats['count'], color='#42A5F5', alpha=0.8,
                   edgecolor='white', width=0.6, label='ICU Stays', zorder=3)

    ax1.set_xlabel('Age Group', fontsize=12)
    ax1.set_ylabel('Number of ICU Stays', fontsize=12)
    ax1.set_xticks(x)
    ax1.set_xticklabels(stats['age_group'], fontsize=11)

    # Mortality overlay
    ax2 = ax1.twinx()
    ax2.plot(x, stats['mortality_pct'], 'o-', color='#F44336',
             linewidth=2.5, markersize=8, label='Mortality Rate', zorder=5)
    ax2.set_ylabel('Mortality Rate (%)', color='#F44336', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='#F44336')

    # Annotate mortality percentages
    for xi, mort in zip(x, stats['mortality_pct']):
        ax2.annotate(f'{mort:.1f}%', (xi, mort),
                     textcoords='offset points', xytext=(0, 12),
                     ha='center', fontsize=10, fontweight='bold', color='#D32F2F')

    ax1.set_title('Mortality Rate by Age Group', fontsize=14, fontweight='bold')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'eda_mortality_by_age.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 12 — Temporal Vital Trends
# ═══════════════════════════════════════════════════════════════════════════════


fig_mortality_by_age(merged)


INFO: Figure 11: Mortality by age group
INFO:   → output\eda\eda_mortality_by_age.png


### Figure 12 — Temporal Vital Trends

In [14]:
def fig_temporal_vitals(loader: MIMICDataLoader, merged: pd.DataFrame, sample_n=2000):
    """Line plots of mean vitals over first 48h: survivors vs non-survivors."""
    logger.info(f"Figure 12: Temporal vital trends (sampling {sample_n} stays)")

    if loader.chartevents is None or len(loader.chartevents) == 0:
        logger.warning("  No chartevents, skipping")
        return

    vitals_to_plot = {
        'Heart Rate': [220045, 211],
        'Mean ABP': [220052, 220181, 52, 456],
        'SpO₂ (%)': [220277, 646],
        'Resp Rate': [220210, 224690, 618, 615],
    }

    # Sample stays, balanced between survivors/deceased
    deceased = merged[merged['expire_flag'] == 1]['icustay_id'].values
    survived = merged[merged['expire_flag'] == 0]['icustay_id'].values

    n_each = min(sample_n // 2, len(deceased), len(survived))
    sampled_dead = np.random.choice(deceased, n_each, replace=False)
    sampled_surv = np.random.choice(survived, n_each, replace=False)
    all_sampled = np.concatenate([sampled_dead, sampled_surv])

    charts = loader.chartevents[loader.chartevents['icustay_id'].isin(all_sampled)].copy()

    # Merge intime for relative time
    intime_map = merged.set_index('icustay_id')['intime'].to_dict()
    expire_map = merged.set_index('icustay_id')['expire_flag'].to_dict()

    charts['intime'] = charts['icustay_id'].map(intime_map)
    charts['expire_flag'] = charts['icustay_id'].map(expire_map)
    charts['hours_since_admit'] = (
        (charts['charttime'] - charts['intime']).dt.total_seconds() / 3600
    )

    # Filter to first 48 hours
    charts = charts[(charts['hours_since_admit'] >= 0) & (charts['hours_since_admit'] <= 48)]

    # Bin into 2-hour windows
    charts['hour_bin'] = (charts['hours_since_admit'] // 2) * 2

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for i, (name, ids) in enumerate(vitals_to_plot.items()):
        ax = axes[i]
        subset = charts[charts['itemid'].isin(ids)].copy()

        if len(subset) == 0:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center')
            ax.set_title(name)
            continue

        for label, flag, color in [('Survived', 0, COLOR_SURV), ('Deceased', 1, COLOR_DEAD)]:
            grp = subset[subset['expire_flag'] == flag]
            if len(grp) == 0:
                continue
            hourly = grp.groupby('hour_bin')['valuenum'].agg(['mean', 'std']).reset_index()

            ax.plot(hourly['hour_bin'], hourly['mean'], '-', color=color,
                    linewidth=2, label=label)
            ax.fill_between(hourly['hour_bin'],
                            hourly['mean'] - hourly['std'] * 0.5,
                            hourly['mean'] + hourly['std'] * 0.5,
                            alpha=0.15, color=color)

        ax.set_xlabel('Hours Since ICU Admission')
        ax.set_ylabel(name)
        ax.set_title(f'{name}: First 48 Hours', fontweight='bold')
        ax.legend(fontsize=10)
        ax.set_xlim(0, 48)
        ax.grid(alpha=0.3)

    plt.suptitle('Temporal Vital Sign Trends: Survivors vs Deceased',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'eda_temporal_vitals.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  FIGURE 13 — Admission Type Distribution
# ═══════════════════════════════════════════════════════════════════════════════


fig_temporal_vitals(loader, merged, sample_n=SAMPLE_N_TEMPORAL)


INFO: Figure 12: Temporal vital trends (sampling 3000 stays)
C:\Users\ncfad\AppData\Local\Temp\ipykernel_26668\3080325396.py:77: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\ncfad\AppData\Local\Temp\ipykernel_26668\3080325396.py:79: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig(path)
INFO:   → output\eda\eda_temporal_vitals.png


### Figure 13 — Admission Type Distribution

In [15]:
def fig_admission_types(merged: pd.DataFrame):
    """Bar chart of admission types with mortality overlay."""
    logger.info("Figure 13: Admission type distribution")

    if 'admission_type' not in merged.columns:
        logger.warning("  No admission_type column, skipping")
        return

    stats = merged.groupby('admission_type').agg(
        count=('subject_id', 'count'),
        mortality=('expire_flag', 'mean'),
    ).reset_index()
    stats['mortality_pct'] = stats['mortality'] * 100
    stats = stats.sort_values('count', ascending=False)

    fig, ax1 = plt.subplots(figsize=(10, 6))

    colors = sns.color_palette('Set2', len(stats))
    bars = ax1.bar(range(len(stats)), stats['count'], color=colors,
                   edgecolor='white', width=0.6, zorder=3)

    ax1.set_xticks(range(len(stats)))
    ax1.set_xticklabels(stats['admission_type'], fontsize=11, rotation=15, ha='right')
    ax1.set_ylabel('Number of ICU Stays')

    for bar, count in zip(bars, stats['count']):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)

    ax2 = ax1.twinx()
    ax2.plot(range(len(stats)), stats['mortality_pct'],
             'D-', color='#F44336', markersize=10, linewidth=2, label='Mortality Rate')
    ax2.set_ylabel('Mortality Rate (%)', color='#F44336')
    ax2.tick_params(axis='y', labelcolor='#F44336')
    ax2.set_ylim(0, max(stats['mortality_pct']) * 1.4)

    for xi, mort in zip(range(len(stats)), stats['mortality_pct']):
        ax2.annotate(f'{mort:.1f}%', (xi, mort),
                     textcoords='offset points', xytext=(0, 10),
                     ha='center', fontsize=10, fontweight='bold', color='#D32F2F')

    ax1.set_title('Admission Type Distribution with Mortality Rate',
                  fontsize=14, fontweight='bold')
    ax2.legend(loc='upper right')

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'eda_admission_types.png')
    plt.savefig(path)
    plt.close()
    logger.info(f"  → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════════════════════


fig_admission_types(merged)


INFO: Figure 13: Admission type distribution
INFO:   → output\eda\eda_admission_types.png


---
## Summary

All **13 EDA figures** have been generated and saved to `output/eda/`.

| Part | Figures | Focus |
|------|---------|-------|
| Part 1 | 1–7 | Demographics, vital/lab distributions |
| Part 2 | 8–13 | Labels, correlations, temporal trends |
